# Modelagem Generativa Baseada em Score

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Em vez de modelar a densidade $p(x)$ diretamente, modelamos o *score* $s_\theta(x) \approx \nabla_x \log p(x)$. Amostragem é feita por dinâmica de Langevin: pequenos passos ruidosos na direção do score levam a regiões de densidade alta.


## Formulação Matemática

$$x_{t+1} = x_t + \tfrac{\epsilon}{2}\, s_\theta(x_t) + \sqrt{\epsilon}\, z_t,\quad z_t \sim \mathcal{N}(0, I)$$

Denoising score matching treina $s_\theta$ via

$$\mathbb{E}_{\sigma, x, \tilde x}\,\left\|s_\theta(\tilde x, \sigma) - \frac{x - \tilde x}{\sigma^2}\right\|^2$$


## Implementação


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [ ]:
# 2-D toy: mixture of two Gaussians
def sample_data(n):
    mix = torch.randint(0, 2, (n,))
    means = torch.tensor([[-2., 0.], [2., 0.]])
    return means[mix] + 0.3 * torch.randn(n, 2)

class ScoreNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.SiLU(),
                                 nn.Linear(64, 64), nn.SiLU(),
                                 nn.Linear(64, 2))
    def forward(self, x, sigma):
        s = sigma.view(-1, 1).expand(-1, 1)
        return self.net(torch.cat([x, s], dim=-1)) / sigma.view(-1, 1)


In [ ]:
sigmas = torch.tensor([2.0, 1.0, 0.5, 0.25, 0.1])

def dsm_loss(model, x):
    s = sigmas[torch.randint(0, len(sigmas), (x.size(0),))]
    noise = torch.randn_like(x)
    x_tilde = x + s.view(-1, 1) * noise
    target = -noise / s.view(-1, 1)
    pred = model(x_tilde, s)
    return ((pred - target) ** 2 * s.view(-1, 1) ** 2).mean()


## Experimento


In [ ]:
model = ScoreNet()
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
for step in range(2000):
    x = sample_data(256)
    loss = dsm_loss(model, x)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f'step {step}  loss {loss.item():.3f}')


In [ ]:
# Annealed Langevin sampling
@torch.no_grad()
def langevin(n=500, steps=100, eps_start=0.1):
    x = torch.randn(n, 2)
    for s in sigmas:
        eps = eps_start * (s.item() / sigmas[-1].item()) ** 2
        for _ in range(steps):
            score = model(x, s.expand(n))
            x = x + eps * score + (2 * eps) ** 0.5 * torch.randn_like(x)
    return x

samples = langevin()
plt.scatter(samples[:, 0], samples[:, 1], s=5)
plt.title('Langevin samples from learnt score'); plt.show()


## Discussão

- Múltiplas escalas (vários $\sigma$) é o que faz funcionar — sigma único é instável.
- A ligação com modelos de difusão: DDPM no limite contínuo é *exatamente* uma SDE de score matching (Song et al.).
- Para dados de alta dimensão (imagens), use uma U-Net e condicionamento idêntico ao DDPM.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
